In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import os
import sys

# --- Correction du chemin d'importation des modules 'src' ---
# Assure que le répertoire parent du projet est dans sys.path pour les imports de src
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    
from src.rag_system import preprocess_query2

print("Modules importés avec succès.")

c:\Users\mohme\Desktop\projet_recettes\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modules importés avec succès.


In [2]:
faiss_index_path = "../data/embeddings/recipe_index.faiss"
metadata_path = "../data/embeddings/recipe_metadata.csv"
model_name = 'all-MiniLM-L6-v2' # Assurez-vous que c'est le même modèle utilisé pour la génération des embeddings

print(f"Chemin de l'index FAISS : {faiss_index_path}")
print(f"Chemin des métadonnées : {metadata_path}")
print(f"Modèle d'embedding utilisé : {model_name}")

Chemin de l'index FAISS : ../data/embeddings/recipe_index.faiss
Chemin des métadonnées : ../data/embeddings/recipe_metadata.csv
Modèle d'embedding utilisé : all-MiniLM-L6-v2


In [3]:
print(f"Chargement du modèle d'embedding '{model_name}'...")
try:
    query_model = SentenceTransformer(model_name)
    print("Modèle chargé avec succès.")
except Exception as e:
    print(f"Erreur lors du chargement du modèle : {e}")
    print("Assurez-vous d'avoir une connexion internet ou que le modèle est mis en cache.")
    sys.exit(1)

Chargement du modèle d'embedding 'all-MiniLM-L6-v2'...
Modèle chargé avec succès.


In [4]:
print("Chargement de l'index FAISS et des métadonnées...")
try:
    index = faiss.read_index(faiss_index_path)
    df_metadata = pd.read_csv(metadata_path)
    print(f"Index FAISS chargé. Nombre de vecteurs : {index.ntotal}")
    print(f"Métadonnées chargées. Nombre de recettes : {len(df_metadata)}")

    # Vérification que l'ordre des métadonnées correspond à l'index FAISS
    if index.ntotal != len(df_metadata):
        print("Avertissement : Le nombre de vecteurs dans l'index FAISS ne correspond pas au nombre de recettes dans les métadonnées.")
        print("Cela pourrait entraîner des résultats incorrects. Veuillez vérifier le processus de génération des embeddings.")

except FileNotFoundError as e:
    print(f"Erreur : Un fichier nécessaire n'a pas été trouvé : {e}")
    print("Assurez-vous d'avoir exécuté le notebook 03_embedding_generation.ipynb et que les chemins sont corrects.")
    sys.exit(1)
except Exception as e:
    print(f"Erreur lors du chargement de l'index ou des métadonnées : {e}")
    sys.exit(1)

Chargement de l'index FAISS et des métadonnées...
Index FAISS chargé. Nombre de vecteurs : 232111
Métadonnées chargées. Nombre de recettes : 232111


In [5]:
print("Chargement de l'index FAISS et des métadonnées...")
try:
    index = faiss.read_index(faiss_index_path)
    df_metadata = pd.read_csv(metadata_path)
    print(f"Index FAISS chargé. Nombre de vecteurs : {index.ntotal}")
    print(f"Métadonnées chargées. Nombre de recettes : {len(df_metadata)}")

    # Vérification que l'ordre des métadonnées correspond à l'index FAISS
    if index.ntotal != len(df_metadata):
        print("Avertissement : Le nombre de vecteurs dans l'index FAISS ne correspond pas au nombre de recettes dans les métadonnées.")
        print("Cela pourrait entraîner des résultats incorrects. Veuillez vérifier le processus de génération des embeddings.")

except FileNotFoundError as e:
    print(f"Erreur : Un fichier nécessaire n'a pas été trouvé : {e}")
    print("Assurez-vous d'avoir exécuté le notebook 03_embedding_generation.ipynb et que les chemins sont corrects.")
    sys.exit(1)
except Exception as e:
    print(f"Erreur lors du chargement de l'index ou des métadonnées : {e}")
    sys.exit(1)

Chargement de l'index FAISS et des métadonnées...
Index FAISS chargé. Nombre de vecteurs : 232111
Métadonnées chargées. Nombre de recettes : 232111


In [6]:
def search_recipes(query, top_k=5):
    """
    Recherche les recettes les plus similaires à une requête donnée.

    Args:
        query (str): La requête textuelle de l'utilisateur.
        top_k (int): Le nombre de recettes les plus similaires à retourner.

    Returns:
        pd.DataFrame: Un DataFrame contenant les recettes les plus similaires.
    """
    # 1. Encoder la requête
    query_embedding = query_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
   

    # 2. Rechercher dans l'index FAISS
   
    distances, indices = index.search(query_embedding, top_k)

    # 3. Récupérer les détails des recettes correspondantes
    
    retrieved_recipes = df_metadata.iloc[indices[0]].copy()

    # Ajouter la distance (ou score de similarité) aux résultats pour information
    retrieved_recipes['faiss_distance'] = distances[0]
    # Tri par distance (les plus petites distances sont les plus pertinentes)
    retrieved_recipes = retrieved_recipes.sort_values(by='faiss_distance').reset_index(drop=True)

    return retrieved_recipes


In [7]:
# Exemple 1 : Recherche générale de tajine
print("--- Recherche : 'recette de tajine de poulet sain' ---")
query1= preprocess_query2("how to make Chicken  Tajine")
print(query1)
results1 = search_recipes(query1, top_k=3)
display(results1[['name', 'tags', 'faiss_distance', 'ingredients_text']]) # Afficher quelques colonnes clés
print("\n")
"""
# Exemple 2 : Recherche de dessert
print("--- Recherche : 'recette facile de gâteau au chocolat' ---")
results2 = search_recipes("recette facile de gâteau au chocolat", top_k=5)
display(results2[['name', 'tags', 'faiss_distance', 'ingredients_text']])
print("\n")

# Exemple 3 : Requête spécifique marocaine
print("--- Recherche : 'comment faire du couscous marocain traditionnel' ---")
results3 = search_recipes("comment faire du couscous marocain traditionnel", top_k=5)
display(results3[['name', 'tags', 'faiss_distance', 'ingredients_text']])
print("\n")"""

--- Recherche : 'recette de tajine de poulet sain' ---
tajine how make to chicken tagine stew moroccan


,name,tags,faiss_distance,ingredients_text
0,moroccan beef stew tagine tajine,"['time-to-make', 'course', 'main-ingredient', ...",0.349613,flour\nsalt\nblack pepper\nbeef stew meat\noil...
1,Moroccan Beef Stew Tagine/Tajine,"['marocaine', 'north african']",0.422087,13 cup flour\n1 14 teaspoons salt\nblack peppe...
2,Moroccan Beef Tajine With Apple & Apricot,"['marocaine', 'north african']",0.450225,2 teaspoon Butter\n2 Onions (chopped)\n4 teasp...


'\n# Exemple 2 : Recherche de dessert\nprint("--- Recherche : \'recette facile de gâteau au chocolat\' ---")\nresults2 = search_recipes("recette facile de gâteau au chocolat", top_k=5)\ndisplay(results2[[\'name\', \'tags\', \'faiss_distance\', \'ingredients_text\']])\nprint("\n")\n\n# Exemple 3 : Requête spécifique marocaine\nprint("--- Recherche : \'comment faire du couscous marocain traditionnel\' ---")\nresults3 = search_recipes("comment faire du couscous marocain traditionnel", top_k=5)\ndisplay(results3[[\'name\', \'tags\', \'faiss_distance\', \'ingredients_text\']])\nprint("\n")'

In [11]:
if not results1.empty:
    print(f"Document complet de la recette '{results1['name'].iloc[2]}' :")
    print(results1['document'].iloc[0])
else:
    print("Aucune recette trouvée pour la troisième recherche.")

Document complet de la recette 'Moroccan Beef Tajine With Apple & Apricot' :

    Title: moroccan beef stew tagine tajine
    Tags: time-to-make, course, main-ingredient, cuisine, preparation, occasion, main-dish, beef, african, easy, dinner-party, moroccan, stove-top, dietary, meat, taste-mood, savory, equipment, 4-hours-or-less
    Ingredients:
    flour
salt
black pepper
beef stew meat
oil
tomatoes
water
cinnamon sticks
ground cumin
orange rind
garlic
sugar
carrots
chickpeas

    Instructions:
    1. combine flour , salt and pepper
2. coat the meat with the flour in a large bowl or large plastic bag
3. in a large skillet or dutch oven brown the meat in very hot oil about 10 minutes or until browned
4. remove from heat
5. stir into the pan the undrained tomatoes , water , cinnamon sticks , orange peel , garlic , cumin and sugar
6. simmer covered about 1 hour 15 minutes or until meat is almost tender
7. add carrots and cook covered until tender
8. add the chick peas and cook until hea